# Gemma 2 9B Fine-tuning on RunPod (A100 Optimized)

이 노트북은 **RunPod A100-SXM** 환경에서 Gemma 2 9B 모델을 파인튜닝하기 위해 최적화되었습니다.

### ⚠️ 해결 방법 (AttributeError: 'Gemma2ForCausalLM' object has no attribute 'set_submodule')
이 에러는 보통 `transformers` 라이브러리 버전이 낮아서 발생합니다. 
1. 아래 첫 번째 셀(**라이브러리 설치**)을 실행하여 최신 버전으로 업데이트하세요.
2. 설치가 완료되면 **반드시** 상단 메뉴에서 `Kernel` -> `Restart Kernel`을 클릭한 뒤 처음부터 다시 실행하세요.

In [ ]:
# 1. 필수 라이브러리 최신 버전 업데이트
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets python-dotenv

In [ ]:
# 2. 임포트 및 로그인
import os
import torch
from huggingface_hub import login
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# RunPod 환경 변수에서 토큰 읽기
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN detected from environment variables.")
    login(token=hf_token)
else:
    hf_token = input("Hugging Face Write Token을 입력하세요: ")
    login(token=hf_token)

In [ ]:
# 3. 데이터셋 로드 및 포맷팅
dataset_path = "/workspace/train_dataset.jsonl"
if not os.path.exists(dataset_path):
    print(f"Error: {dataset_path} not found! Please upload your dataset to /workspace/")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")

    def format_gemma_prompt(examples):
        instructions = examples["instruction"]
        inputs = examples["input"]
        outputs = examples["output"]
        texts = []
        for instruction, input_text, output in zip(instructions, inputs, outputs):
            text = (
                f"<start_of_turn>user\n"
                f"{instruction}\n\n"
                f"질문: {input_text}<end_of_turn>\n"
                f"<start_of_turn>model\n"
                f"{output}<end_of_turn>"
            )
            texts.append(text)
        return {"text": texts}

    dataset = dataset.map(format_gemma_prompt, batched=True)
    print("Dataset loaded and formatted.")

In [ ]:
# 4. 모델 로드 (A100 최적화)
model_id = "google/gemma-2-9b-it"
output_dir = "/workspace/gemma2-pet-counselor-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    # A100에서 속도를 대폭 높여주는 Flash Attention 2 활성화
    # 만약 에러 발생 시 아래 줄을 주석 처리 하세요.
    # attn_implementation="flash_attention_2",
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

# 1. PEFT(LoRA) 설정
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=32,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 2. SFTConfig 설정
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    bf16=True,           
    fp16=False,
    max_grad_norm=0.3,
    warmup_steps=100,     
    lr_scheduler_type="cosine",
    report_to="none"
)

# 3. 트레이너 초기화 (tokenizer -> processing_class 변경)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer, # 💡 바로 이 부분입니다!
    args=sft_config,
)

# 4. 학습 시작
print("🚀 [System] All hurdles cleared. Training starting now on A100...")
trainer.train()

# 5. 저장
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ [Success] Model saved to {output_dir}")